# Лабораторная работа: Методы обучения без учителя
### Кластеризация и снижение размерности

**Датасет:** Iris (встроенный датасет scikit-learn)  
**Задача:** Снижение размерности (PCA, t-SNE) + кластеризация (K-Means, DBSCAN, AgglomerativeClustering)

## 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import warnings
warnings.filterwarnings('ignore')

print('Все библиотеки успешно импортированы!')

## 2. Загрузка и подготовка датасета

Используем датасет **Iris** — классический датасет с данными об ирисах.  
Он содержит 150 образцов и 4 признака:
- sepal length (длина чашелистика)
- sepal width (ширина чашелистика)
- petal length (длина лепестка)
- petal width (ширина лепестка)

**Датасет D1** — все 4 признака без целевой переменной (метки класса).

In [ ]:
# Загрузка датасета
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
y_true = iris.target  # истинные метки (для сравнения)

print('Размер датасета:', df.shape)
print('\nПервые 5 строк:')
df.head()

In [ ]:
# Датасет D1 — все 4 признака (без целевой переменной)
D1 = df.values.copy()

# Масштабирование признаков — важный шаг перед кластеризацией
scaler = StandardScaler()
D1_scaled = scaler.fit_transform(D1)

print('D1 — датасет с 4 признаками, размер:', D1_scaled.shape)
print('Среднее после масштабирования (должно быть ~0):', D1_scaled.mean(axis=0).round(2))

## 3. Снижение размерности — PCA

**PCA (Метод главных компонент)** — линейный метод. Он находит новые оси (главные компоненты), 
вдоль которых данные разбросаны максимально сильно. Таким образом, мы сжимаем 4 признака до 2, 
сохраняя как можно больше информации.

In [ ]:
# Применяем PCA — снижение размерности до 2
pca = PCA(n_components=2, random_state=42)
D2 = pca.fit_transform(D1_scaled)

print('D2 — датасет после PCA, размер:', D2.shape)
print(f'Объясненная дисперсия по компонентам: {pca.explained_variance_ratio_.round(3)}')
print(f'Итого объяснено дисперсии: {pca.explained_variance_ratio_.sum():.1%}')

## 4. Снижение размерности — t-SNE

**t-SNE** — нелинейный метод. Он пытается сохранить локальную структуру данных: 
точки, которые близки в исходном пространстве, должны оставаться близкими на 2D-плоскости.

In [ ]:
# Применяем t-SNE — снижение размерности до 2
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
D3 = tsne.fit_transform(D1_scaled)

print('D3 — датасет после t-SNE, размер:', D3.shape)

## 5. Визуализация D2 (PCA) и D3 (t-SNE)

Нарисуем оба датасета и покрасим точки в соответствии с **истинными метками класса** 
(которые мы специально не использовали при обучении, но применяем только для визуализации).

In [ ]:
colors = ['#e74c3c', '#2ecc71', '#3498db']
class_names = ['Setosa', 'Versicolor', 'Virginica']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- PCA ---
ax = axes[0]
for i, name in enumerate(class_names):
    mask = y_true == i
    ax.scatter(D2[mask, 0], D2[mask, 1], c=colors[i], label=name, alpha=0.8, s=60)
ax.set_title('D2: PCA (2 компоненты)', fontsize=13)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(True, alpha=0.3)

# --- t-SNE ---
ax = axes[1]
for i, name in enumerate(class_names):
    mask = y_true == i
    ax.scatter(D3[mask, 0], D3[mask, 1], c=colors[i], label=name, alpha=0.8, s=60)
ax.set_title('D3: t-SNE (2 компоненты)', fontsize=13)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Сравнение методов снижения размерности', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('dim_reduction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Рисунок сохранён: dim_reduction.png')

**Вывод по визуализации:**  
На обоих графиках видно три группы. PCA разделяет Setosa хорошо, но Versicolor и Virginica частично перекрываются. t-SNE разделяет все три класса более чётко — кластеры компактнее и лучше изолированы.

## 6. Кластеризация

Применим три метода кластеризации к каждому из датасетов D1, D2, D3.  
Используем **k=3**, так как знаем что в датасете 3 класса (в реальной задаче можно подобрать методом локтя).

**Методы:**
- **K-Means** — разбивает данные на k кластеров по центройдам
- **DBSCAN** — ищет плотные области, умеет определять выбросы
- **AgglomerativeClustering** — иерархический метод, снизу вверх объединяет точки

In [ ]:
def run_clustering(data, data_name):
    """Запускает три метода кластеризации и считает метрики"""
    results = {}
    
    methods = {
        'K-Means': KMeans(n_clusters=3, random_state=42, n_init=10),
        'DBSCAN': DBSCAN(eps=0.8, min_samples=5),
        'AgglomerativeClustering': AgglomerativeClustering(n_clusters=3)
    }
    
    print(f'\n{'='*60}')
    print(f'Датасет: {data_name}')
    print(f'{'='*60}')
    
    for name, model in methods.items():
        labels = model.fit_predict(data)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Метрики считаем только если кластеров > 1
        if n_clusters > 1 and n_clusters < len(data):
            sil = silhouette_score(data, labels)
            db = davies_bouldin_score(data, labels)
            ch = calinski_harabasz_score(data, labels)
        else:
            sil, db, ch = None, None, None
        
        results[name] = {'labels': labels, 'silhouette': sil, 'davies_bouldin': db,
                         'calinski_harabasz': ch, 'n_clusters': n_clusters}
        
        print(f'\n  {name}:')
        print(f'    Найдено кластеров: {n_clusters}', end='')
        if n_noise > 0:
            print(f' (шумовых точек: {n_noise})', end='')
        print()
        if sil is not None:
            print(f'    Silhouette Score:       {sil:.4f}  (выше — лучше, макс. 1.0)')
            print(f'    Davies-Bouldin Index:   {db:.4f}  (ниже — лучше)')
            print(f'    Calinski-Harabasz:      {ch:.2f}  (выше — лучше)')
        else:
            print('    Метрики недоступны (недостаточно кластеров)')
    
    return results

# Запуск для всех датасетов
results_D1 = run_clustering(D1_scaled, 'D1 (исходные 4 признака)')
results_D2 = run_clustering(D2, 'D2 (PCA, 2 компоненты)')
results_D3 = run_clustering(D3, 'D3 (t-SNE, 2 компоненты)')

## 7. Визуализация результатов кластеризации

In [ ]:
def plot_clustering_results(data_2d, results, title_prefix):
    methods = list(results.keys())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    cmap = matplotlib.colormaps.get_cmap('tab10')

    for ax, method in zip(axes, methods):
        labels = results[method]['labels']
        unique = sorted(set(labels))
        for lbl in unique:
            mask = labels == lbl
            color = 'grey' if lbl == -1 else cmap(lbl)
            lname = 'Шум' if lbl == -1 else f'Кластер {lbl+1}'
            ax.scatter(data_2d[mask, 0], data_2d[mask, 1], c=[color]*mask.sum(),
                       label=lname, alpha=0.8, s=50)
        
        sil = results[method]['silhouette']
        sil_str = f'\nSilhouette={sil:.3f}' if sil else ''
        ax.set_title(f'{method}{sil_str}', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Кластеризация: {title_prefix}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'clustering_{title_prefix.lower().replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Для D2 и D3 можем построить напрямую (они уже 2D)
plot_clustering_results(D2, results_D2, 'D2 PCA')
plot_clustering_results(D3, results_D3, 'D3 t-SNE')

# Для D1 используем D2 для визуализации (проекция)
plot_clustering_results(D2, results_D1, 'D1 (визуализация через PCA)')

## 8. Сводная таблица метрик

In [ ]:
all_results = {
    'D1 (4 признака)': results_D1,
    'D2 (PCA)': results_D2,
    'D3 (t-SNE)': results_D3
}

rows = []
for dataset_name, res in all_results.items():
    for method, metrics in res.items():
        rows.append({
            'Датасет': dataset_name,
            'Метод': method,
            'Кластеров': metrics['n_clusters'],
            'Silhouette ↑': round(metrics['silhouette'], 4) if metrics['silhouette'] else '-',
            'Davies-Bouldin ↓': round(metrics['davies_bouldin'], 4) if metrics['davies_bouldin'] else '-',
            'Calinski-Harabasz ↑': round(metrics['calinski_harabasz'], 2) if metrics['calinski_harabasz'] else '-',
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 9. Выводы

**По снижению размерности:**
- **PCA** сохраняет ~97% дисперсии данных в 2 компонентах. Линейный метод, быстрый и интерпретируемый. Хорошо разделяет Setosa, но Versicolor и Virginica частично перекрываются.
- **t-SNE** лучше разделяет все три класса визуально — кластеры более компактны и изолированы. Нелинейный метод, не сохраняет глобальные расстояния.

**По кластеризации:**
- **K-Means** показывает стабильно хорошие результаты на всех датасетах. Работает хорошо, когда кластеры примерно одинакового размера и шарообразной формы.
- **DBSCAN** менее подходит для этого датасета — плотности кластеров неоднородны. Показывает нестабильные результаты в зависимости от параметров.
- **AgglomerativeClustering** также показывает хорошие результаты, сопоставимые с K-Means.

**Лучший результат** по метрике Silhouette, как правило, достигается на датасете D3 (t-SNE) с алгоритмом K-Means или AgglomerativeClustering, поскольку t-SNE делает кластеры компактнее.